In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import pyfar as pf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display
from algorithms.avendano import ambiance_index, panning_index

In [ ]:
hrirs, sources = pf.signals.files.head_related_impulse_responses(position='horizontal', diffuse_field_compensation=True)

In [ ]:
elevation = 0
azimuth_L = np.pi/3 # 60 degrees left
azimuth_R = -np.pi/3 # 60 degrees right
radius = 1

desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)

index_L, _ = sources.find_nearest(desired_direction_L)
index_R, _ = sources.find_nearest(desired_direction_R)

sources.show([index_L, index_R])
plt.show()


In [ ]:
ax = pf.plot.freq_group_delay(hrirs[index_R], label=["Left ear", "Right ear"])
# ax[0].legend()
# ax[1].legend()
plt.show()

In [ ]:
kick = pf.io.read_audio(Path.home() / "Music/freesound/581451__johnnydekk__fractanimal_acoustic_drum_kit_kick_2.wav")
hihat = pf.io.read_audio(Path.home() / "Music/freesound/581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav")
# ax_kick = pf.plot.time_freq(kick)
ax_hihat = pf.plot.time_freq(hihat)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(4,4), sharex=True, layout='tight')

def render(audio, hrir, ax):
    binaural_audio = pf.dsp.convolve(audio, hrir)
    cropped = pf.dsp.time_crop(binaural_audio, [0,0.03], unit='s')
    cropped.time[0] += 0.5
    cropped.time[1] -= 0.5
    pf.plot.time(cropped, label=["Left ear", "Right ear"], ax=ax)
    ax.set_ylim(-1.5,1.5)

render(hihat, hrirs[index_R], ax[0])
render(kick, hrirs[index_R], ax[1])
plt.show()

In [ ]:
music = pf.io.read_audio("/Users/richard/Music/archive/09. Tsubasao Kudasai.flac", stop=int(10. * 44100))
pf.plot.spectrogram(music, freq_scale='log')
plt.show()

In [ ]:
from matplotlib.colors import LogNorm

def amplitude_pan(sig: pf.Signal, pan: float) -> pf.Signal:
    assert -1 <= pan <= 1.
    assert sig.cshape == (2,)
    a = (pan + 1) / 2
    al, ar = np.cos(a * 0.5 * np.pi), np.sin(a * 0.5 * np.pi)
    print(a, al, ar)
    left, right = sig.time[0] * al, sig.time[1] * ar
    return pf.Signal(np.stack([left, right]), sig.sampling_rate)

def downmix(sig: pf.Signal) -> pf.Signal:
    assert sig.cshape == (2,)
    a = 1/np.sqrt(2)
    data = np.sum(sig.time, axis=0) * a
    return pf.Signal(np.stack([data, data]), sig.sampling_rate)

def make_panned_sound(pan: float = 0.5):
    f1 = "/Users/richard/Music/freesound/474398__bangcorrupt__blues_guitar_intro.flac"
    f2 = "/Users/richard/Music/freesound/42956__freqman__violin_tchaikovsky_danse-arabe.wav"
    sound1 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f1), sampling_rate=48000), [0.,10.], unit='s')), -pan)
    sound2 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f2), sampling_rate=48000), [0.,10.], unit='s')), pan)
    sound = sound1 + sound2
    return sound

def get_music():
    f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
    return pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f), sampling_rate=48000), [0.,10.], unit='s')

# sound = get_music()
sound = make_panned_sound(pan=0.6)

freqs, times, spect = pf.dsp.spectrogram(sound)
spect = np.swapaxes(spect, 1, 2)

fig, ax = plt.subplots(3,1, figsize=(6,12))

ai = ambiance_index(spect, lamb=0.85)
pi = panning_index(spect)

panogram = np.vstack([np.histogram(m, bins=100, range=[-1,1])[0] for m in pi])
ax[0].imshow(np.rot90(ai), aspect='auto', cmap='grey')
ax[1].imshow(np.rot90(pi), aspect='auto', cmap='berlin')
ax[2].imshow(np.rot90(panogram), aspect='auto', cmap='grey', norm=LogNorm())
print(np.min(pi), np.max(pi))


Audio(sound.time, rate=sound.sampling_rate)